# 03 · Model training

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| **3. Model training** | **this notebook** |
| 4. Evaluation | `04_evaluation.ipynb` |

This notebook **recomputes** training through `aerocast.pipeline`, the same functions `scripts/run_phase1.py` calls. At the end it checks that it reproduces the script's saved artefacts in `results/`. The script remains the single source of truth for the reported numbers. Runtime is about 2 minutes.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd
from aerocast import config, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
R = config.RESULTS_DIR

## 1. Chronological split

No shuffling: every model is trained on the past and evaluated on the future, exactly as it would be deployed.

In [2]:
P = pipeline.prepare()
pd.DataFrame(P.split_summary()).T.assign(role=["fitting + rolling-origin CV", "calibration + threshold choice", "touched once, final numbers"])

,rows,delay_rate,from,to,role
train,211684,0.2649,2013-01-08,2013-08-31,fitting + rolling-origin CV
calib,55628,0.157,2013-09-01,2013-10-31,calibration + threshold choice
test,53991,0.2583,2013-11-01,2013-12-31,"touched once, final numbers"


* **Jan 1–7 excluded** (warm-up): history windows are empty or very thin.
* **Cancelled/diverted flights excluded** from the target, but they were counted in the congestion features (notebook 02).
* Note the **base-rate shift**: 26.5% in training, 15.7% in calibration, 25.8% in test (Nov 18%, **Dec 33.5%**).

## 2. Baselines

Any learned model must beat:
* **prior**: predict the training base rate for every flight (PR-AUC = base rate, ROC-AUC = 0.5);
* **route history**: the smoothed 28-day carrier-route delay rate on its own, i.e. "this flight on this route is usually late".

In [3]:
raw_preds = pipeline.baseline_preds(P)
pd.DataFrame({m: {s: float(p.mean()) for s, p in d.items()} for m, d in raw_preds.items()}).T.add_prefix("mean score · ")

,mean score · train,mean score · calib,mean score · test
prior,0.2649,0.2649,0.2649
route_history,0.2638,0.1615,0.2227


## 3. Why rolling-origin CV instead of one validation month

The first design tuned on **September** alone (train Jan–Aug → validate Sep). It failed in two ways:
* XGBoost early-stopped after about **13 trees**, and its September score **did not track** later months.
* September is the **calmest month of the year** (13.7% late). PR-AUC depends on the base rate, so one calm month is a poor stand-in for later months.

The diagnostic below shows this with the final hyperparameters. It fits XGBoost on Jan–Aug and plots PR-AUC on September and on October as trees are added. **No test data is used.**

In [4]:
S = json.loads((R / "summary.json").read_text())
curve = pipeline.single_month_tree_curve(P, S["xgb_params"])
fig = plots.tree_count_curve(curve, S["xgb_params"]["n_estimators"])
plots.save(fig, "train_single_month_tree_curve"); fig

<Figure size 650x320 with 1 Axes>

In [5]:
pd.DataFrame({m: {"best n_trees": int(curve.loc[curve[c].idxmax(), "n_trees"]), "best PR-AUC": curve[c].max(),
                   "PR-AUC range": curve[c].max() - curve[c].min()} for m, c in [("Sep", "pr_auc_m9"), ("Oct", "pr_auc_m10")]}).T

,best n_trees,best PR-AUC,PR-AUC range
Sep,110.0,0.2437,0.0147
Oct,50.0,0.2678,0.0184


Two **adjacent** months disagree on the optimal number of trees by about 2× (110 vs 50), and both curves are nearly flat (a range of only 0.015–0.018 PR-AUC). A single month is a noisy guide for early stopping and model selection. Note also the absolute level: about 0.24–0.27 here versus about 0.47 in the summer CV folds. That gap is the base-rate effect again, not a difference in model quality.

**Rolling-origin CV** inside the training window fixes this. For each validation month m ∈ {May, Jun, Jul, Aug}: train on all months before m, validate on m.

| Fold | Train | Validate |
|---|---|---|
| 1 | Jan–Apr | May |
| 2 | Jan–May | Jun |
| 3 | Jan–Jun | Jul |
| 4 | Jan–Jul | Aug |

Hyperparameters are chosen on the **mean of four seasonal folds**. Training never sees a month after the one it is validated on (asserted in `tests/`).

## 4. Logistic regression

Pipeline: one-hot encoding of the IDs (`min_frequency=50`) + median imputation **with missing-value indicators** + standardisation → L2 logistic regression. No `class_weight`, so the probabilities stay calibrated. Grid: C ∈ {0.0003, 0.001, 0.003, 0.01, 0.1, 1}.

In [6]:
lr = pipeline.train_logreg(P)
wide = lr.cv.pivot(index="C", columns="fold", values="val_pr_auc")
wide.assign(mean=wide.mean(axis=1), std=wide.std(axis=1))

fold,0,1,2,3,mean,std
C,,,,,,
0.0003,0.4425,0.5277,0.5547,0.4082,0.4833,0.0692
0.0010,0.4427,0.5266,0.5576,0.4080,0.4837,0.0700
0.0030,0.4416,0.5230,0.5584,0.4080,0.4828,0.0698
0.0100,0.4391,0.5188,0.5582,0.4075,0.4809,0.0697
0.1000,0.4352,0.5141,0.5575,0.4068,0.4784,0.0696
1.0000,0.4349,0.5133,0.5572,0.4065,0.4780,0.0695


In [7]:
print("chosen C =", lr.C)
fig = plots.cv_logreg_curve(lr.cv); plots.save(fig, "train_cv_logreg"); fig

chosen C = 0.001


<Figure size 600x320 with 1 Axes>

**C = 0.001**, i.e. strong regularisation. Its neighbours on both sides score lower, so the optimum is inside the grid, not at its edge. The fold-to-fold spread (May/Aug ≈ 0.40 vs Jun/Jul ≈ 0.53) is much larger than the differences between C values. That spread is the base-rate effect again: June and July have more delays, so PR-AUC is higher.

## 5. XGBoost

* 12 configurations sampled from a grid of depth {2, 3, 5} × min_child_weight {20, 100} × colsample {0.5, 0.8} × λ {1, 10}, with learning rate 0.03.
* For each configuration and fold: up to 3,000 trees, **early stopping** (100 rounds) on that fold's validation month.
* Score = mean fold PR-AUC. **Tree count = mean of the four best iterations**, which is then used to refit on all of Jan–Aug *without* early stopping, so no data outside training decides when to stop.
* Carrier, origin and destination are handled with native categorical splits.

In [8]:
xgb = pipeline.train_xgb(P, n_configs=12)
xgb.cv_configs

,mean_pr_auc,std_pr_auc,n_trees
config,,,
3,0.4695,0.0742,294
5,0.4688,0.0743,267
2,0.4684,0.0719,213
4,0.4682,0.0728,239
11,0.4674,0.0746,361
9,0.4673,0.0759,278
8,0.4665,0.0719,306
0,0.4654,0.0722,317
6,0.4652,0.0722,356


In [9]:
best_cfg = xgb.cv_configs.index[0]
xgb.cv_folds[xgb.cv_folds.config == best_cfg][["fold", "best_iteration", "val_pr_auc"]]

,fold,best_iteration,val_pr_auc
12,0,260,0.4074
13,1,231,0.5041
14,2,514,0.5578
15,3,170,0.4088


In [10]:
pd.Series(xgb.params, name="chosen").to_frame()

,chosen
max_depth,3.00
learning_rate,0.03
min_child_weight,100.00
subsample,0.80
colsample_bytree,0.50
reg_lambda,1.00
n_estimators,294.00


The search picks **shallow, heavily constrained trees** (depth 3, min_child_weight = 100). Deeper trees overfit, and the top configurations score within about 0.002 of each other. Like the C = 0.001 result for LogReg, this says the day-ahead signal is weak and mostly additive, so controlling variance matters more than model flexibility.

In [11]:
cs = pipeline.cv_summary(lr, xgb)
pd.DataFrame({"mean CV PR-AUC": [cs["logreg"]["mean"], cs["xgboost"]["mean_pr_auc"]],
              "std across folds": [cs["logreg"]["std"], cs["xgboost"]["std_pr_auc"]]}, index=["logreg", "xgboost"])

,mean CV PR-AUC,std across folds
logreg,0.4837,0.0700
xgboost,0.4695,0.0742


LogReg's mean CV score is slightly higher than XGBoost's, but the difference (about 0.014) is far smaller than the fold-to-fold standard deviation (about 0.07). On the training window the two are tied, and notebook 04 tests this formally on the test period.

## 6. Final models (fit on all of Jan–Aug)

Both `train_*` functions have already refit the selected configuration on the full training window. Their scores on every split feed calibration and evaluation in notebook 04.

In [12]:
raw_preds["logreg"] = pipeline.predict_splits(lr.model, P)
raw_preds["xgboost"] = pipeline.predict_splits(xgb.model, P)
pd.DataFrame({m: {s: float(p.mean()) for s, p in d.items()} for m, d in raw_preds.items()}).T.add_prefix("mean score · ")

,mean score · train,mean score · calib,mean score · test
prior,0.2649,0.2649,0.2649
route_history,0.2638,0.1615,0.2227
logreg,0.2649,0.2136,0.2501
xgboost,0.2649,0.2208,0.2253


The mean raw score is close to the training base rate (0.265) on every split. The models haven't yet been told that Sep–Oct is calmer. Calibration (notebook 04) is where that information could enter.

## 7. Consistency with `scripts/run_phase1.py`

This notebook recomputed everything from scratch. The checks below confirm it matches the artefacts the script wrote, so notebook 04 (which reads those artefacts) is evaluating exactly these models.

In [13]:
pd.testing.assert_frame_equal(lr.cv.reset_index(drop=True), pd.read_csv(R / "cv_logreg.csv"))
pd.testing.assert_frame_equal(xgb.cv_configs, pd.read_csv(R / "cv_xgb_configs.csv", index_col="config"))
assert lr.C == S["logreg_C"] and xgb.params == S["xgb_params"]

_, saved_raw = pipeline.load_predictions()
for m in pipeline.RAW_SCORES:
    for s in config.SPLITS:
        assert np.array_equal(raw_preds[m][s].astype(saved_raw[m][s].dtype), saved_raw[m][s]), (m, s)
print("OK: CV tables, chosen hyper-parameters and all predictions match results/")

OK: CV tables, chosen hyper-parameters and all predictions match results/
